# Notebook 06 — Final Decision & Utility-based Threshold Selection (Project 3: Sepsis EWS)

This notebook is the **deployment decision layer**. It **does not retrain models**.  
It loads the saved artifacts from Notebook 04/05 and:

1. **Reproduces a constraint-based operating point** (τ\*) for **Policy A**  
2. Computes an **utility-optimal operating point** (τᵤ) using a cost/benefit-inspired utility  
3. Saves **final, interview-proof artifacts**:
   - `results/final_decision.json`
   - `results/final_operating_point.csv`
   - `results/policyA_selected_utility.json`

> Recommended preset: **Clinical / High-detection utility (Preset A)**  
because missed sepsis (FN) is usually far more costly than additional alerts.


In [4]:

# Imports & paths
from pathlib import Path
import json
import numpy as np
import pandas as pd

DATA_ROOT = Path(r"F:\Project 3 Folder\project 3--ews")
RESULTS_DIR = DATA_ROOT / "results"
FIG_DIR = DATA_ROOT / "figures"
RESULTS_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

A_PATH = RESULTS_DIR / "threshold_sweep_policyA_v3.csv"
B_PATH = RESULTS_DIR / "threshold_sweep_policyB_v3.csv"   # optional (ablation)
CALIB_PATH = RESULTS_DIR / "calibration_report.json"
OOF_PATH = RESULTS_DIR / "oof_with_calibration.parquet"

print("RESULTS_DIR:", RESULTS_DIR.resolve())


RESULTS_DIR: F:\Project 3 Folder\project 3--ews\results


In [5]:

# Load artifacts
assert A_PATH.exists(), f"Missing {A_PATH}. Run Notebook 04 to generate v3 Policy A sweep."

sweepA = pd.read_csv(A_PATH)
print("Loaded Policy A sweep:", A_PATH.name, sweepA.shape)

sweepB = pd.read_csv(B_PATH) if B_PATH.exists() else None
print("Loaded Policy B sweep:", B_PATH.exists())

assert CALIB_PATH.exists(), f"Missing {CALIB_PATH}. Run Notebook 04 to generate calibration_report.json"
calib_report = json.loads(CALIB_PATH.read_text())
print("Calibration:", calib_report.get("model_name", "?"), "| chosen:", calib_report.get("chosen", calib_report.get("calibrator", "?")))

assert OOF_PATH.exists(), f"Missing {OOF_PATH}. Run Notebook 04 to generate oof_with_calibration.parquet"
oof = pd.read_parquet(OOF_PATH)
print("Loaded OOF:", oof.shape)

reqA = {"tau","sepsis_detection_rate","nonsepsis_alert_rate","alerts_per_100_patient_hours","lead_time_median"}
missing = reqA - set(sweepA.columns)
assert not missing, f"Policy A sweep missing columns: {missing}"
print("Sanity checks passed ✅")


Loaded Policy A sweep: threshold_sweep_policyA_v3.csv (147, 16)
Loaded Policy B sweep: True
Calibration: hgb | chosen: isotonic
Loaded OOF: (19145, 9)
Sanity checks passed ✅


## 1) Constraint-based operating point (τ\*)

**Low-alarm operating point** (deployment default):
- `nonsepsis_alert_rate <= 0.10`
- `alerts_per_100_patient_hours <= 1.0`
- objective: **maximize median lead time**, tie-breaker: **maximize detection**, then **minimize burden**


In [6]:

MAX_FALSE_ALERT = 0.10
MAX_ALERTS_PER_100H = 1.0

cand = sweepA[
    (sweepA["nonsepsis_alert_rate"] <= MAX_FALSE_ALERT) &
    (sweepA["alerts_per_100_patient_hours"] <= MAX_ALERTS_PER_100H)
].copy()

if cand.empty:
    raise ValueError("No thresholds satisfy constraints. Relax constraints or verify sweep columns.")

cand = cand.sort_values(
    ["lead_time_median", "sepsis_detection_rate", "alerts_per_100_patient_hours"],
    ascending=[False, False, True]
)

tau_star = float(cand.iloc[0]["tau"])
row_star = cand.iloc[0].to_dict()

print("Constraint tau* =", tau_star)
for k in ["sepsis_detection_rate","nonsepsis_alert_rate","alerts_per_100_patient_hours","lead_time_median"]:
    print(f"{k}: {row_star.get(k)}")


Constraint tau* = 0.0854430379746835
sepsis_detection_rate: 0.8974358974358975
nonsepsis_alert_rate: 0.0
alerts_per_100_patient_hours: 0.8670671193523114
lead_time_median: 40.0


## 2) Utility-based selection (τᵤ)

### Presets
- **Preset A (recommended): Clinical / High-detection**
- Preset B: Balanced / Alarm-fatigue aware


In [7]:

def compute_utility(df: pd.DataFrame,
                    w_det: float,
                    w_lead: float,
                    w_burden: float,
                    w_false: float,
                    lead_cap_hours: float = 96.0,
                    burden_norm: float = 2.5):
    out = df.copy()

    lead = out["lead_time_median"].astype(float).fillna(0.0).clip(lower=0.0)
    lead_norm = lead.clip(upper=lead_cap_hours) / lead_cap_hours

    burden = out["alerts_per_100_patient_hours"].astype(float).fillna(burden_norm)
    burden_normed = burden.clip(lower=0.0, upper=burden_norm) / burden_norm

    false_rate = out["nonsepsis_alert_rate"].astype(float).fillna(1.0).clip(lower=0.0)
    det = out["sepsis_detection_rate"].astype(float).fillna(0.0).clip(0.0, 1.0)

    out["utility"] = (w_det * det) + (w_lead * lead_norm) - (w_burden * burden_normed) - (w_false * false_rate)
    return out

PRESET_A = dict(name="A_clinical_high_detection", w_det=4.0, w_lead=1.5, w_burden=1.0, w_false=2.0)
PRESET_B = dict(name="B_balanced_alarm_fatigue", w_det=3.0, w_lead=1.0, w_burden=2.0, w_false=2.0)

preset = PRESET_A  # recommended
print("Using preset:", preset["name"])


Using preset: A_clinical_high_detection


In [8]:

uA = compute_utility(
    sweepA,
    w_det=preset["w_det"],
    w_lead=preset["w_lead"],
    w_burden=preset["w_burden"],
    w_false=preset["w_false"],
    lead_cap_hours=96.0,
    burden_norm=2.5
)

uA = uA.sort_values(["utility", "sepsis_detection_rate", "lead_time_median"], ascending=[False, False, False])

tau_u = float(uA.iloc[0]["tau"])
row_u = uA.iloc[0].to_dict()

print("Utility tau_u =", tau_u)
for k in ["utility","sepsis_detection_rate","nonsepsis_alert_rate","alerts_per_100_patient_hours","lead_time_median"]:
    print(f"{k}: {row_u.get(k)}")


Utility tau_u = 0.0791139240506329
utility: 4.203329034996083
sepsis_detection_rate: 1.0
nonsepsis_alert_rate: 0.0
alerts_per_100_patient_hours: 1.6401149125097936
lead_time_median: 55.0


In [9]:

# Compare tau* vs tau_u
import numpy as np
import pandas as pd

def pack(d, tag):
    return {
        "choice": tag,
        "tau": float(d.get("tau")),
        "det": float(d.get("sepsis_detection_rate", np.nan)),
        "false_pat": float(d.get("nonsepsis_alert_rate", np.nan)),
        "burden_per100h": float(d.get("alerts_per_100_patient_hours", np.nan)),
        "lead_med_h": float(d.get("lead_time_median", np.nan)),
        "utility": float(d.get("utility", np.nan))
    }

pd.DataFrame([pack(row_star, "constraint_tau_star"), pack(row_u, f"utility_{preset['name']}")])


,choice,tau,det,false_pat,burden_per100h,lead_med_h,utility
0,constraint_tau_star,0.085443,0.897436,0.0,0.867067,40.0,NaN
1,utility_A_clinical_high_detection,0.079114,1.000000,0.0,1.640115,55.0,4.203329


## 3) Save final artifacts

- `results/policyA_selected_utility.json`
- `results/final_decision.json`
- `results/final_operating_point.csv`

Decision rule:
Prefer τᵤ unless it drops detection by more than 1%.


In [11]:

import json, numpy as np, pandas as pd

out_utility = RESULTS_DIR / "policyA_selected_utility.json"
out_final_json = RESULTS_DIR / "final_decision.json"
out_final_csv = RESULTS_DIR / "final_operating_point.csv"

policyA_utility = {
    "preset": preset,
    "policy": "A_first_crossing_lockout",
    "lockout_hours": int(6),
    "tau_u": tau_u,
    "metrics": {k: row_u.get(k) for k in ["sepsis_detection_rate","nonsepsis_alert_rate","alerts_per_100_patient_hours","lead_time_median","utility"]},
    "comparison_tau_star": {"tau_star": tau_star, "metrics": {k: row_star.get(k) for k in ["sepsis_detection_rate","nonsepsis_alert_rate","alerts_per_100_patient_hours","lead_time_median"]}}
}
out_utility.write_text(json.dumps(policyA_utility, indent=2))
print("Saved:", out_utility)

DET_DROP_TOL = 0.01
det_star = float(row_star.get("sepsis_detection_rate", 0.0))
det_u = float(row_u.get("sepsis_detection_rate", 0.0))
use_utility = det_u >= (det_star - DET_DROP_TOL)

final_tau = tau_u if use_utility else tau_star
final_row = row_u if use_utility else row_star
final_choice = f"utility_{preset['name']}" if use_utility else "constraint_tau_star"

final_decision = {
    "model_name": calib_report.get("model_name", "hgb"),
    "calibrator": calib_report.get("chosen", calib_report.get("calibrator", "isotonic")),
    "policy": "A_first_crossing_lockout",
    "lockout_hours": int(6),
    "operating_point_choice": final_choice,
    "tau": float(final_tau),
    "metrics": {
        "sepsis_detection_rate": float(final_row.get("sepsis_detection_rate", np.nan)),
        "nonsepsis_alert_rate": float(final_row.get("nonsepsis_alert_rate", np.nan)),
        "alerts_per_100_patient_hours": float(final_row.get("alerts_per_100_patient_hours", np.nan)),
        "lead_time_median": float(final_row.get("lead_time_median", np.nan)),
    }
}
out_final_json.write_text(json.dumps(final_decision, indent=2))
print("Saved:", out_final_json)

pd.DataFrame([{
    "model": final_decision["model_name"],
    "calibrator": final_decision["calibrator"],
    "policy": final_decision["policy"],
    "operating_point_choice": final_decision["operating_point_choice"],
    "tau": final_decision["tau"],
    **final_decision["metrics"]
}]).to_csv(out_final_csv, index=False)
print("Saved:", out_final_csv)

print("=== FINAL SYSTEM (COPY FOR README) ===")
print(f"Model: {final_decision['model_name']} | Calibrator: {final_decision['calibrator']}")
print(f"Policy: {final_decision['policy']} (lockout={final_decision['lockout_hours']}h)")
print(f"Operating point: {final_decision['operating_point_choice']} | tau={final_decision['tau']:.5f}")
m = final_decision["metrics"]
print(f"Detection (sepsis): {m['sepsis_detection_rate']:.3f}")
print(f"False-alert patients (non-sepsis): {m['nonsepsis_alert_rate']:.5f}")
print(f"Alarm burden: {m['alerts_per_100_patient_hours']:.3f} alerts / 100 patient-hours")
print(f"Median lead time: {m['lead_time_median']:.1f} hours")


Saved: F:\Project 3 Folder\project 3--ews\results\policyA_selected_utility.json
Saved: F:\Project 3 Folder\project 3--ews\results\final_decision.json
Saved: F:\Project 3 Folder\project 3--ews\results\final_operating_point.csv
=== FINAL SYSTEM (COPY FOR README) ===
Model: hgb | Calibrator: isotonic
Policy: A_first_crossing_lockout (lockout=6h)
Operating point: utility_A_clinical_high_detection | tau=0.07911
Detection (sepsis): 1.000
False-alert patients (non-sepsis): 0.00000
Alarm burden: 1.640 alerts / 100 patient-hours
Median lead time: 55.0 hours
